<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-04/AI_Modul_4.5_Praktikum_Exploratory_Data_Analysis.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 4.5: Praktikum Exploratory Data Analysis (EDA)
## Investigasi Tiga Tingkat: Momen Distribusi, Korelasi Non-Linier, dan Paradoks Simpson

Notebook ini menyajikan implementasi praktis metodologi **Exploratory Data Analysis (EDA)** menurut paradigma John Tukey untuk membedah karakteristik data produksi kelapa sawit, mendeteksi multikolinieritas, serta membongkar fenomena **Paradoks Simpson** pada respon pemupukan.

### Agenda Praktikum:
1. **Inisialisasi Lingkungan Komputasi & Import Pustaka Analitik**
2. **Pembangkitan Dataset Perkebunan Multi-Faktor (Sawit Mineral & Gambut)**
3. **Tingkat 1: Analisis Univariat (Mean, Median, Skewness, Kurtosis, & KDE)**
4. **Tingkat 2: Analisis Bivariat (Uji Komparasi Pearson vs Spearman)**
5. **Tingkat 3: Analisis Multivariat & Matriks Korelasi Heatmap**
6. **Investigasi Khusus: Eksperimen & Visualisasi Paradoks Simpson**
7. **Ekstraksi Wawasan Agronomi untuk Kesiapan Pemodelan AI**

In [ ]:
# 1. Inisialisasi Environment & Import Pustaka
import os
os.environ['MPLBACKEND'] = 'Agg'  # Backend non-interaktif

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print(f"Versi Pandas : {pd.__version__}")
print(f"Versi Seaborn: {sns.__version__}")
print("Lingkungan EDA siap digunakan!")

## 2. Pembangkitan Dataset Multi-Faktor Agribisnis Kelapa Sawit

Kita menyimulasikan 300 blok kebun yang tersebar pada tanah mineral dan gambut dengan karakteristik agronomi riil.

In [ ]:
np.random.seed(42)
N = 300

blok_id = [f"BLK-{i+1:03d}" for i in range(N)]
tipe_tanah = np.random.choice(['Mineral', 'Gambut'], size=N, p=[0.6, 0.4])
umur = np.random.randint(6, 22, size=N)

# Pemupukan KCl (kg/pokok): Pada tanah gambut secara historis dipupuk lebih banyak
pupuk_kcl = np.where(
    tipe_tanah == 'Gambut',
    np.random.uniform(3.5, 6.0, size=N),
    np.random.uniform(1.0, 3.8, size=N)
).round(2)

# Respon Kadar Minyak OER (%): Non-linier saturasi diminishing returns terhadap KCl
# Potensi biologis dasar tanah mineral (23%) lebih tinggi dari gambut (19%)
oer_dasar = np.where(tipe_tanah == 'Mineral', 21.0, 17.5)
oer = oer_dasar + 2.8 * np.log(pupuk_kcl + 0.5) + np.random.normal(0, 0.6, size=N)
oer = np.clip(oer, 16.0, 27.5).round(2)

# Curah Hujan & NDVI
curah_hujan = np.random.gamma(shape=3.0, scale=60.0, size=N).round(1) # Distribusi right-skewed
ndvi = np.clip(0.50 + 0.05 * np.log(curah_hujan + 10) + np.random.normal(0, 0.04, size=N), 0.45, 0.90).round(3)

# Tonase TBS (Ton/Ha) dengan Paradoks Simpson yang sengaja dibentuk:
# Di dalam masing-masing tanah, pupuk berkorelasi POSITIF terhadap yield.
# Namun potensi hasil gambut (rerata 18 Ton) < mineral (rerata 28 Ton).
yield_tbs = np.where(
    tipe_tanah == 'Mineral',
    24.0 + 1.8 * pupuk_kcl + 6.0 * ndvi + np.random.normal(0, 1.2, size=N),
    14.0 + 1.4 * pupuk_kcl + 4.5 * ndvi + np.random.normal(0, 1.2, size=N)
).round(2)

df_sawit = pd.DataFrame({
    'Blok_ID': blok_id,
    'Tipe_Tanah': tipe_tanah,
    'Umur_Thn': umur,
    'Pupuk_KCl_kg': pupuk_kcl,
    'Curah_Hujan_mm': curah_hujan,
    'Indeks_NDVI': ndvi,
    'Kadar_OER_pct': oer,
    'Tonase_TBS_Ton_Ha': yield_tbs
})

print("Tampilan 5 Baris Pertama Data Kebun Sawit:")
print(df_sawit.head())

## 3. Tingkat 1: Analisis Univariat (Momen Distribusi)

Menghitung 4 momen distribusi statistik (Mean, Median, Skewness, Kurtosis) pada variabel `Curah_Hujan_mm` dan `Tonase_TBS_Ton_Ha`.

In [ ]:
def analisis_momen(seri, nama):
    m1_mean = seri.mean()
    m1_median = seri.median()
    m2_std = seri.std()
    m3_skew = stats.skew(seri)
    m4_kurt = stats.kurtosis(seri)  # Fisher excess kurtosis (Normal = 0)
    
    tipe_skew = "Menceng Kanan (Right-Skewed)" if m3_skew > 0.5 else ("Menceng Kiri (Left-Skewed)" if m3_skew < -0.5 else "Simetris")
    tipe_kurt = "Leptokurtik (Fat-Tail)" if m4_kurt > 0.5 else ("Platikurtik (Thin-Tail)" if m4_kurt < -0.5 else "Mesokurtik (Normal)")
    
    print(f"=== Profil Distribusi: {nama} ===")
    print(f"• Rerata (Mean)    : {m1_mean:.2f}")
    print(f"• Median           : {m1_median:.2f}")
    print(f"• Deviasi Standar  : {m2_std:.2f}")
    print(f"• Skewness (g1)    : {m3_skew:.2f} -> {tipe_skew}")
    print(f"• Excess Kurt (g2) : {m4_kurt:.2f} -> {tipe_kurt}\n")

analisis_momen(df_sawit['Curah_Hujan_mm'], 'Curah Hujan Bulanan (mm)')
analisis_momen(df_sawit['Tonase_TBS_Ton_Ha'], 'Produksi Tonase TBS (Ton/Ha)')

# Visualisasi Univariat Histogram + KDE + Boxplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

sns.histplot(df_sawit['Curah_Hujan_mm'], kde=True, color='#2563EB', ax=ax1, bins=25)
ax1.axvline(df_sawit['Curah_Hujan_mm'].mean(), color='red', ls='--', label='Rerata')
ax1.axvline(df_sawit['Curah_Hujan_mm'].median(), color='green', ls='-', label='Median')
ax1.set_title('Distribusi Curah Hujan (Right-Skewed Gamma)', fontsize=10, fontweight='bold')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.5)

sns.boxplot(y='Tonase_TBS_Ton_Ha', x='Tipe_Tanah', hue='Tipe_Tanah', legend=False, data=df_sawit, palette=['#10B981', '#F59E0B'], ax=ax2)
ax2.set_title('Distribusi Bivariat Tonase per Tipe Tanah', fontsize=10, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_5_distribusi.png', dpi=150)
print("Plot distribusi univariat disimpan ke docs/assets/praktikum_4_5_distribusi.png")

## 4. Tingkat 2: Analisis Bivariat (Pearson vs Spearman)

Membandingkan korelasi linier Pearson dengan korelasi peringkat Spearman pada kurva respon saturasi biologis (Dosis KCl vs Kadar Minyak OER).

In [ ]:
# Uji Korelasi Dosis KCl vs Kadar OER pada Blok Tanah Mineral
df_min = df_sawit[df_sawit['Tipe_Tanah'] == 'Mineral'].copy()

r_pearson, p_pearson = stats.pearsonr(df_min['Pupuk_KCl_kg'], df_min['Kadar_OER_pct'])
r_spearman, p_spearman = stats.spearmanr(df_min['Pupuk_KCl_kg'], df_min['Kadar_OER_pct'])

print("=== Komparasi Korelasi Respon Biologis Pupuk KCl vs OER ===")
print(f"Korelasi Linier Pearson   (r)  : {r_pearson:.4f} (p-value: {p_pearson:.2e})")
print(f"Korelasi Monotonik Spearman (rs): {r_spearman:.4f} (p-value: {p_spearman:.2e})")
print("Interpretasi: Spearman lebih tinggi karena menangkap relasi saturasi kurvilinier logaritmik secara tepat!")

## 5. Tingkat 3: Analisis Multivariat & Matriks Korelasi

Menganalisis interaksi antar-seluruh fitur numerik menggunakan Heatmap Korelasi.

In [ ]:
fitur_numerik = ['Umur_Thn', 'Pupuk_KCl_kg', 'Curah_Hujan_mm', 'Indeks_NDVI', 'Kadar_OER_pct', 'Tonase_TBS_Ton_Ha']
corr_mat = df_sawit[fitur_numerik].corr(method='spearman')

plt.figure(figsize=(8, 6), dpi=150)
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1.0, vmax=1.0, square=True, linewidths=0.5)
plt.title('Matriks Korelasi Spearman Multivariat Kebun Sawit', fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_5_korelasi.png', dpi=150)
print("Plot heatmap multivariat disimpan ke docs/assets/praktikum_4_5_korelasi.png")

## 6. Investigasi Khusus: Pembuktian Empiris Paradoks Simpson

Kita membuktikan bagaimana korelasi agregat global antara `Pupuk_KCl_kg` dan `Tonase_TBS_Ton_Ha` berbalik arah secara dramatis saat didekomposisi per `Tipe_Tanah`.

In [ ]:
# 1. Korelasi Global Seluruh Kebun (Mengabaikan Tipe Tanah)
r_global, _ = stats.pearsonr(df_sawit['Pupuk_KCl_kg'], df_sawit['Tonase_TBS_Ton_Ha'])

# 2. Korelasi Lokal di Masing-masing Tipe Tanah
df_gambut = df_sawit[df_sawit['Tipe_Tanah'] == 'Gambut']
r_mineral, _ = stats.pearsonr(df_min['Pupuk_KCl_kg'], df_min['Tonase_TBS_Ton_Ha'])
r_gambut, _ = stats.pearsonr(df_gambut['Pupuk_KCl_kg'], df_gambut['Tonase_TBS_Ton_Ha'])

print("=== PEMBUKTIAN PARADOKS SIMPSON PADA REKOMENDASI PEMUPUKAN ===")
print(f"• Korelasi Global (Seluruh Kebun)  : r = {r_global:.3f} (SEOLAH-OLAH NEGATIF!)")
print(f"• Korelasi Lokal (Tanah Mineral)   : r = +{r_mineral:.3f} (POSITIF KUAT)")
print(f"• Korelasi Lokal (Tanah Gambut)    : r = +{r_gambut:.3f} (POSITIF KUAT)")

# 3. Visualisasi Pembalikan Gradien Paradoks Simpson
plt.figure(figsize=(9, 5.5), dpi=150)
sns.scatterplot(x='Pupuk_KCl_kg', y='Tonase_TBS_Ton_Ha', hue='Tipe_Tanah',
                palette={'Mineral': '#10B981', 'Gambut': '#D97706'}, data=df_sawit, alpha=0.7, s=50)

# Garis Regresi Global (Mencerminkan tren semu negatif)
sns.regplot(x='Pupuk_KCl_kg', y='Tonase_TBS_Ton_Ha', data=df_sawit, scatter=False,
            color='#DC2626', line_kws={'linestyle': '--', 'linewidth': 2.5, 'label': f'Tren Global (Salah: r = {r_global:.2f})'})

# Garis Regresi Lokal Mineral
sns.regplot(x='Pupuk_KCl_kg', y='Tonase_TBS_Ton_Ha', data=df_min, scatter=False,
            color='#047857', line_kws={'linewidth': 2, 'label': f'Tren Mineral (Benar: r = +{r_mineral:.2f})'})

# Garis Regresi Lokal Gambut
sns.regplot(x='Pupuk_KCl_kg', y='Tonase_TBS_Ton_Ha', data=df_gambut, scatter=False,
            color='#B45309', line_kws={'linewidth': 2, 'label': f'Tren Gambut (Benar: r = +{r_gambut:.2f})'})

plt.title('Visualisasi Paradoks Simpson: Tren Pemupukan KCl terhadap Produksi TBS', fontsize=11, fontweight='bold', pad=12)
plt.xlabel('Dosis Pupuk KCl (kg/pokok)', fontsize=9, fontweight='bold')
plt.ylabel('Produksi TBS (Ton/Ha)', fontsize=9, fontweight='bold')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('e:/Project Buku/docs/assets/praktikum_4_5_simpson.png', dpi=150)
print("Plot Paradoks Simpson berhasil disimpan ke docs/assets/praktikum_4_5_simpson.png")
print("\nPraktikum Modul 4.5 Selesai dengan Sukses!")